# Hypothesis Testing — Python Code Reference

Practical Python implementations of the concepts from the theory document. Run these cells, modify the parameters, and see how the math plays out in practice.

---

## Contents

1. Setup
2. Basic sample statistics
3. Visualizing the √n effect (CLT)
4. One-sample z-test (σ known)
5. Two-sided z-test
6. Two-proportion z-test
7. Using SciPy's built-in functions
8. Confidence interval helper
9. Reusable hypothesis-test function
10. Visualizing a hypothesis test
11. Monte Carlo simulation
12. Quick reference cheat sheet

## 1. Setup

Import the core scientific Python stack.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)  # reproducible results

## 2. Basic Sample Statistics

Computing the sample mean, variance, and standard deviation — first from scratch, then the NumPy way.

Formulas:

$$\bar{X} = \frac{1}{n}\sum_{i=1}^{n} X_i \qquad S^2 = \frac{1}{n-1}\sum_{i=1}^{n}(X_i - \bar{X})^2 \qquad S = \sqrt{S^2}$$

> ⚠️ Always use `ddof=1` for the sample standard deviation. The default `ddof=0` divides by `n` and gives a biased estimator.

In [ ]:
# Example: 36 pizza delivery times (minutes)
times = np.array([30, 25, 28, 35, 22, 27, 31, 26, 29, 33, 24, 28,
                  26, 30, 27, 32, 25, 28, 29, 27, 31, 23, 28, 30,
                  26, 29, 27, 34, 25, 28, 30, 26, 29, 27, 31, 28])

n = len(times)

# --- From scratch ---
X_bar = sum(times) / n
variance = sum((x - X_bar)**2 for x in times) / (n - 1)  # Bessel's correction
S = variance ** 0.5

print(f"n       = {n}")
print(f"X_bar   = {X_bar:.4f}")
print(f"S^2     = {variance:.4f}")
print(f"S       = {S:.4f}")

# --- With NumPy (the practical way) ---
X_bar_np = np.mean(times)
S_np     = np.std(times, ddof=1)   # ddof=1 → divide by (n-1)

print(f"\nNumPy:  X_bar = {X_bar_np:.4f},  S = {S_np:.4f}")

## 3. Visualizing the √n Effect (CLT in Action)

The Central Limit Theorem says: no matter how weirdly shaped the population is, the distribution of the **sample mean** X̄ becomes normal as n grows, and its spread shrinks as σ/√n.

Below we draw samples from a skewed **exponential** distribution and plot the distribution of X̄ for increasing n. Watch the histograms narrow and get more bell-shaped.

In [ ]:
# Generate a skewed population (exponential distribution)
population = np.random.exponential(scale=5, size=100_000)
mu_true    = population.mean()
sigma_true = population.std()

sample_sizes = [1, 5, 30, 200]
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)

for ax, n in zip(axes, sample_sizes):
    # Draw 5000 samples of size n, compute the mean of each
    sample_means = [np.random.choice(population, size=n).mean()
                    for _ in range(5000)]
    ax.hist(sample_means, bins=40, density=True,
            color='steelblue', edgecolor='white')
    ax.axvline(mu_true, color='red', linestyle='--', label=f'μ = {mu_true:.2f}')
    ax.set_title(f'n = {n}\nSE ≈ {sigma_true/np.sqrt(n):.2f}')
    ax.set_xlabel('Sample mean')
    ax.legend()

axes[0].set_ylabel('Density')
plt.suptitle('CLT: distribution of X̄ tightens as n grows', fontsize=13)
plt.tight_layout()
plt.show()

## 4. One-Sample z-test (σ known)

**Coffee shop example:** the manager claims the average wait time is μ₀ = 4.0 min, with known σ = 1.2. You sample 50 orders and get X̄ = 4.35. Has the wait time increased?

Hypotheses:
- H₀: μ = 4.0
- H_A: μ > 4.0 (one-sided, right-tailed)

Test statistic:

$$z = \frac{\bar{X} - \mu_0}{\sigma/\sqrt{n}}$$

In [ ]:
# Given data
mu_0  = 4.0       # null-hypothesis mean
sigma = 1.2       # known population std
n     = 50
X_bar = 4.35      # observed sample mean
alpha = 0.05

# Step 1: standard error
SE = sigma / np.sqrt(n)

# Step 2: z-statistic
z = (X_bar - mu_0) / SE

# Step 3: p-values
p_one_sided = 1 - stats.norm.cdf(z)
p_two_sided = 2 * (1 - stats.norm.cdf(abs(z)))

# Step 4: critical values
z_crit_one = stats.norm.ppf(1 - alpha)        # 1.645
z_crit_two = stats.norm.ppf(1 - alpha/2)      # 1.96

print(f"SE                 = {SE:.4f}")
print(f"z                  = {z:.4f}")
print(f"p-value (one-sided) = {p_one_sided:.4f}")
print(f"p-value (two-sided) = {p_two_sided:.4f}")
print(f"critical z (one)   = ±{z_crit_one:.3f}")
print(f"critical z (two)   = ±{z_crit_two:.3f}")

if p_one_sided < alpha:
    print(f"\n✅ Reject H₀ at α={alpha}: evidence wait times increased")
else:
    print(f"\n❌ Fail to reject H₀: no evidence of increase")

## 5. Two-Sided z-test

**NBA scoring example:** last season μ₀ = 94.81, σ = 7.16. After a rule change, 432 games average X̄ = 94.69. Has scoring changed — in *either* direction?

Hypotheses:
- H₀: μ = 94.81
- H_A: μ ≠ 94.81 (two-sided)

In [ ]:
mu_0  = 94.81
sigma = 7.16
n     = 432
X_bar = 94.69
alpha = 0.05

SE = sigma / np.sqrt(n)
z  = (X_bar - mu_0) / SE
p  = 2 * (1 - stats.norm.cdf(abs(z)))   # two-sided p-value

# 95% confidence interval
ci_lo = X_bar - 1.96 * SE
ci_hi = X_bar + 1.96 * SE

print(f"SE      = {SE:.4f}")
print(f"z       = {z:.4f}")
print(f"p-value = {p:.4f}")
print(f"95% CI  = [{ci_lo:.2f}, {ci_hi:.2f}]")
print(f"μ₀ in CI? {'yes' if ci_lo <= mu_0 <= ci_hi else 'no'}")

if p < alpha:
    direction = "increased" if z > 0 else "decreased"
    print(f"\n✅ Reject H₀: scoring {direction}")
else:
    print(f"\n❌ Fail to reject H₀: no evidence of change")

## 6. Two-Proportion z-test

**Salk vaccine trial (1954):** is the polio rate in the vaccinated group lower than in placebo?

- Vaccine group: 57 cases out of 200,000
- Placebo group: 142 cases out of 200,000

Hypotheses:
- H₀: p₁ = p₂
- H_A: p₁ < p₂ (one-sided, left-tailed)

Under H₀, both groups share the same true rate, so we **pool** them to estimate SE:

$$\hat{p} = \frac{x_1 + x_2}{n_1 + n_2} \qquad \text{SE} = \sqrt{\hat{p}(1-\hat{p})\left(\frac{1}{n_1} + \frac{1}{n_2}\right)}$$

In [ ]:
# Given data
x1, n1 = 57,  200_000    # vaccine group
x2, n2 = 142, 200_000    # placebo group
alpha = 0.05

# Sample proportions
p1_hat = x1 / n1
p2_hat = x2 / n2

# Pooled proportion (assumed under H₀)
p_pool = (x1 + x2) / (n1 + n2)

# Standard error
SE = np.sqrt(p_pool * (1 - p_pool) * (1/n1 + 1/n2))

# z-statistic
z = (p1_hat - p2_hat) / SE

# One-sided (left-tailed) p-value
p_value = stats.norm.cdf(z)

# Effect sizes
absolute_reduction = p2_hat - p1_hat
relative_reduction = (p2_hat - p1_hat) / p2_hat

print(f"p1_hat             = {p1_hat:.6f}  ({x1/n1*100:.4f}%)")
print(f"p2_hat             = {p2_hat:.6f}  ({x2/n2*100:.4f}%)")
print(f"pooled p           = {p_pool:.6f}")
print(f"SE                 = {SE:.6e}")
print(f"z                  = {z:.4f}")
print(f"p-value            = {p_value:.3e}")
print(f"absolute reduction = {absolute_reduction:.6f}")
print(f"relative reduction = {relative_reduction*100:.1f}%")

if p_value < alpha:
    print(f"\n✅ Reject H₀ decisively: vaccine works")

## 7. Using SciPy's Built-in Functions

For real work, prefer the library functions — they handle edge cases and give clean output.

In [ ]:
# --- One-sample t-test (σ unknown — the realistic case) ---
data = np.random.normal(loc=4.3, scale=1.2, size=50)
t_stat, p_val = stats.ttest_1samp(data, popmean=4.0)
print(f"One-sample t-test:  t = {t_stat:.3f},  p = {p_val:.4f}")

# One-sided version
p_one = p_val / 2 if t_stat > 0 else 1 - p_val / 2
print(f"One-sided p-value:  {p_one:.4f}")

# --- Two-sample t-test (Welch's, unequal variances) ---
group_a = np.random.normal(10, 2, size=30)
group_b = np.random.normal(11, 2, size=30)
t_stat, p_val = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"\nTwo-sample t-test:  t = {t_stat:.3f},  p = {p_val:.4f}")

In [ ]:
# --- Two-proportion z-test via statsmodels ---
# (If you don't have statsmodels: pip install statsmodels)
from statsmodels.stats.proportion import proportions_ztest

count   = np.array([57, 142])
nobs    = np.array([200_000, 200_000])
z_stat, p_val = proportions_ztest(count, nobs, alternative='smaller')
print(f"Two-proportion z-test:  z = {z_stat:.3f},  p = {p_val:.3e}")

## 8. Confidence Interval Helper

A reusable function that builds a CI for any confidence level.

$$\text{CI} = \bar{X} \pm z^* \cdot \text{SE}$$

Higher confidence → wider interval. You trade precision for certainty.

In [ ]:
def confidence_interval(X_bar, SE, confidence=0.95):
    """
    Build a confidence interval for a mean.

    Parameters
    ----------
    X_bar : float       Sample mean
    SE    : float       Standard error
    confidence : float  e.g. 0.95 for 95% CI

    Returns
    -------
    (lo, hi) : tuple of floats
    """
    alpha  = 1 - confidence
    z_crit = stats.norm.ppf(1 - alpha/2)
    margin = z_crit * SE
    return X_bar - margin, X_bar + margin


ci_95 = confidence_interval(4.35, 0.1697, confidence=0.95)
ci_99 = confidence_interval(4.35, 0.1697, confidence=0.99)
print(f"95% CI: [{ci_95[0]:.3f}, {ci_95[1]:.3f}]")
print(f"99% CI: [{ci_99[0]:.3f}, {ci_99[1]:.3f}]")

## 9. Reusable Hypothesis-Test Function

Wrap the whole pipeline into one utility for fast experimentation.

In [ ]:
def z_test_mean(X_bar, mu_0, sigma, n, alpha=0.05, alternative='two-sided'):
    """
    One-sample z-test for a mean (σ known).

    alternative : 'two-sided', 'greater', or 'less'
    """
    SE = sigma / np.sqrt(n)
    z  = (X_bar - mu_0) / SE

    if alternative == 'two-sided':
        p      = 2 * (1 - stats.norm.cdf(abs(z)))
        z_crit = stats.norm.ppf(1 - alpha/2)
        reject = abs(z) >= z_crit
    elif alternative == 'greater':
        p      = 1 - stats.norm.cdf(z)
        z_crit = stats.norm.ppf(1 - alpha)
        reject = z >= z_crit
    elif alternative == 'less':
        p      = stats.norm.cdf(z)
        z_crit = -stats.norm.ppf(1 - alpha)
        reject = z <= z_crit
    else:
        raise ValueError("alternative must be 'two-sided', 'greater', or 'less'")

    ci_lo, ci_hi = confidence_interval(X_bar, SE, 1 - alpha)

    return {
        'z': z,
        'SE': SE,
        'p_value': p,
        'z_critical': z_crit,
        'reject_H0': reject,
        'CI': (ci_lo, ci_hi),
    }


# Run all three examples through the same function
print("Coffee shop (one-sided, greater):")
for k, v in z_test_mean(4.35, 4.0, 1.2, 50, alternative='greater').items():
    print(f"  {k:<12} = {v}")

print("\nNBA (two-sided):")
for k, v in z_test_mean(94.69, 94.81, 7.16, 432, alternative='two-sided').items():
    print(f"  {k:<12} = {v}")

## 10. Visualizing a Hypothesis Test

A reusable plotting function that draws the null distribution, the rejection region, and where the observed statistic lands.

In [ ]:
def plot_z_test(X_bar, mu_0, sigma, n, alpha=0.05,
                alternative='two-sided', xlabel='Sample mean'):
    SE    = sigma / np.sqrt(n)
    z_obs = (X_bar - mu_0) / SE
    x     = np.linspace(mu_0 - 4*SE, mu_0 + 4*SE, 500)
    y     = stats.norm.pdf(x, loc=mu_0, scale=SE)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(x, y, color='navy', lw=2, label='Null distribution')

    # Rejection region
    if alternative == 'two-sided':
        z_crit = stats.norm.ppf(1 - alpha/2)
        lo, hi = mu_0 - z_crit*SE, mu_0 + z_crit*SE
        ax.fill_between(x[x <= lo], 0, stats.norm.pdf(x[x <= lo], mu_0, SE),
                        color='orange', alpha=0.4)
        ax.fill_between(x[x >= hi], 0, stats.norm.pdf(x[x >= hi], mu_0, SE),
                        color='orange', alpha=0.4, label=f'Rejection α={alpha}')
    elif alternative == 'greater':
        z_crit = stats.norm.ppf(1 - alpha)
        hi = mu_0 + z_crit*SE
        ax.fill_between(x[x >= hi], 0, stats.norm.pdf(x[x >= hi], mu_0, SE),
                        color='orange', alpha=0.4, label=f'Rejection α={alpha}')
    elif alternative == 'less':
        z_crit = stats.norm.ppf(1 - alpha)
        lo = mu_0 - z_crit*SE
        ax.fill_between(x[x <= lo], 0, stats.norm.pdf(x[x <= lo], mu_0, SE),
                        color='orange', alpha=0.4, label=f'Rejection α={alpha}')

    ax.axvline(mu_0,  color='black', ls='--', alpha=0.6, label=f'μ₀ = {mu_0}')
    ax.axvline(X_bar, color='red',   lw=2.5, label=f'X̄ = {X_bar} (z={z_obs:.2f})')
    ax.set_xlabel(xlabel); ax.set_ylabel('Density')
    ax.set_title(f'Hypothesis test ({alternative})')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


# Coffee shop: one-sided right-tailed
plot_z_test(X_bar=4.35, mu_0=4.0, sigma=1.2, n=50,
            alternative='greater', xlabel='Wait time (min)')

In [ ]:
# NBA: two-sided
plot_z_test(X_bar=94.69, mu_0=94.81, sigma=7.16, n=432,
            alternative='two-sided', xlabel='Points per game')

## 11. Monte Carlo Simulation

Rather than relying on the theoretical null distribution, we can **simulate** what happens under H₀ and compute an empirical p-value. This is a powerful sanity check — the two should agree within simulation noise.

In [ ]:
def monte_carlo_z_test(X_bar_obs, mu_0, sigma, n, n_sim=100_000,
                       alternative='greater'):
    """Simulate n_sim samples under H₀ and compute an empirical p-value."""
    # Sample means under H₀ follow N(mu_0, sigma/√n) by the CLT
    sim_means = np.random.normal(loc=mu_0, scale=sigma/np.sqrt(n), size=n_sim)

    if alternative == 'greater':
        p_emp = np.mean(sim_means >= X_bar_obs)
    elif alternative == 'less':
        p_emp = np.mean(sim_means <= X_bar_obs)
    else:  # two-sided
        diff_obs = abs(X_bar_obs - mu_0)
        p_emp    = np.mean(np.abs(sim_means - mu_0) >= diff_obs)

    return p_emp


# Compare theoretical vs empirical for the coffee shop
z = (4.35 - 4.0) / (1.2 / np.sqrt(50))
p_theoretical = 1 - stats.norm.cdf(z)
p_empirical   = monte_carlo_z_test(4.35, mu_0=4.0, sigma=1.2, n=50,
                                   n_sim=200_000, alternative='greater')

print(f"Theoretical p-value : {p_theoretical:.4f}")
print(f"Empirical p-value   : {p_empirical:.4f}")

## 12. Quick Reference Cheat Sheet

Critical z-values for common significance levels, and two-way conversion between z and p.

In [ ]:
print("Critical z-values")
print("-----------------")
print(f"α = 0.10:  one-sided z* = {stats.norm.ppf(0.90):.4f}")
print(f"α = 0.05:  one-sided z* = {stats.norm.ppf(0.95):.4f}")
print(f"α = 0.05:  two-sided z* = {stats.norm.ppf(0.975):.4f}")
print(f"α = 0.01:  one-sided z* = {stats.norm.ppf(0.99):.4f}")
print(f"α = 0.01:  two-sided z* = {stats.norm.ppf(0.995):.4f}")

print("\np-value from z")
print("--------------")
for z in [1.0, 1.645, 1.96, 2.06, 2.58, 3.0]:
    p1 = 1 - stats.norm.cdf(z)
    p2 = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"z = {z:.3f}:  one-sided p = {p1:.4f},  two-sided p = {p2:.4f}")

## Tips for Experimentation

- **Change the seed** at the top (`np.random.seed(...)`) to see how randomness affects conclusions.
- **Increase n** and watch SE shrink, the CI tighten, and p-values drop.
- **Decrease the effect size** (move X̄ closer to μ₀) and see significance disappear.
- **Try different α** values (0.001, 0.10) to see how the threshold changes verdicts.
- **Compare z-test and t-test results** on the same data — they converge as n grows.

The goal is to stop treating these formulas as ceremonial incantations and start *playing* with them. Once you can feel how each parameter nudges the outcome, hypothesis testing becomes intuition, not ritual.

---

## Installation

```bash
pip install numpy scipy matplotlib pandas statsmodels
```